#### 1. Infrastructure: Audit Table Initialization
This section handles the Foundational Layer of your data profiling strategy. It ensures that a persistent Delta table exists to store quality metrics across all pipeline runs.

**Purpose:**
- Persistence: Uses the Delta Lake format to ensure metrics survive cluster restarts.

- Schema Enforcement: Establishes a strict schema to record structure, content, and relationship metadata.
 
- Temporal Tracking: Includes job_timestamp and audit_date to allow for time-series analysis of data quality.

**Technical Logic:**

The function initialize_audit_table uses an **IF NOT EXISTS SQL** clause. This makes the pipeline idempotent, meaning it can be run multiple times without failing or creating duplicate tables.

In [0]:
from pyspark.sql.utils import AnalysisException

def initialize_audit_table(table_name="data_landing.audit.profiling_summary"):
    """
    Creates the profiling audit table if it does not exist.
    
    Args:
        table_name (str): Fully qualified name of the audit table.
    """
    create_query = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
      profile_run_id STRING,
      job_timestamp TIMESTAMP,
      audit_date TIMESTAMP,
      table_name STRING,
      column_name STRING,
      data_type_expected STRING,
      data_type_actual STRING,
      row_count LONG,
      null_count LONG,
      null_percentage FLOAT,
      unique_count LONG,
      min_value STRING,
      max_value STRING,
      mean_value FLOAT,
      is_fk_valid BOOLEAN
    ) USING DELTA
    """
    try:
        spark.sql(create_query)
        print(f"Audit table '{table_name}' initialized.")
    except Exception as e:
        print(f"Failed to initialize table: {str(e)}")

### 2. Profiling Engine: Metric Calculation
The Profiling Engine is the "brain" of the quality check. It analyzes the data at a columnar level to identify potential issues before they reach downstream consumers.

**Features**
- **Structure Discovery:** Captures row_count and validates expected vs. actual data types.

- **Content Assessment:** Calculates null_count, null_percentage, and unique_count.

- **RAM Efficiency:** Processes columns using Spark's distributed architecture, avoiding the OOM (Out of Memory) issues common with Pandas.

**Key Metrics Defined**:
- unique_count: Essential for verifying that primary keys (like RegionName or Unique_City_ID) are truly unique.

- null_percentage: The primary "Quality Gate" metric used to determine if a dataset is fit for purpose.

In [0]:
from datetime import datetime
from pyspark.sql import functions as F

def generate_profiling_metrics(df, dataset_label, run_id):
    """
    Calculates summary statistics for each column in the DataFrame.
    
    Args:
        df (DataFrame): The Spark DataFrame to analyze.
        dataset_label (str): Name of the dataset for logging.
        run_id (str): Unique ID for the current job run.
        
    Returns:
        list: Collection of tuples containing metrics for each column.
    """
    current_ts = datetime.now()
    results = []
    
    for col_name in df.columns:
        stats = df.select(
            F.count("*").alias("total"),
            F.count(F.when(F.col(col_name).isNull(), col_name)).alias("nulls"),
            F.countDistinct(col_name).alias("uniques")
        ).collect()[0]
        
        row = (
            run_id, current_ts, current_ts, dataset_label, col_name,
            str(df.schema[col_name].dataType), str(df.schema[col_name].dataType),
            stats['total'], stats['nulls'], 
            (stats['nulls'] / stats['total'] * 100) if stats['total'] > 0 else 0.0,
            stats['uniques'], None, None, 0.0, True
        )
        results.append(row)
    return results

### 3. Orchestration: The Pipeline Wrapper
This is the Execution Layer. It connects the raw data stored in your Databricks Volumes to the Profiling Engine and the Audit Table.

**Operational Workflow**
- **Lazy Loading**: Data is read using Spark's inferSchema to capture initial data types.

- **Metric Aggregation**: The engine processes the data and returns a structured list of metrics.

- **Audit Logging**: Metrics are appended to the global profiling_summary table.

- **Memory Management**: Explicitly triggers Python's Garbage Collector (gc.collect()) to clear memory after each dataset is processed.

**Usage Pattern**
This orchestrator is designed to handle Single Files or Chunked Folders. Simply pass the Volume path and a descriptive label to the profile_dataset function to include it in the audit.

In [0]:
import gc
import uuid

def profile_dataset_dynamically(base_vol, ds_folder, audit_table="data_landing.audit.profiling_summary"):
    """
    Resolves dataset paths and writes results after ensuring the audit table exists.
    """
    # 1. Check if table exists; if not, initialize it
    table_exists = spark.catalog.tableExists(audit_table)
    if not table_exists:
        print(f"Table {audit_table} not found. Initializing...")
        initialize_audit_table(audit_table)
    
    run_id = str(uuid.uuid4())
    root_path = f"{base_vol}/{ds_folder}"
    
    try:
        # 2. Dynamic Path Resolution
        files = dbutils.fs.ls(root_path)
        if any(f.name == 'chunks/' for f in files):
            final_path = f"{root_path}/chunks/chunk1.csv"
        else:
            final_path = [f.path for f in files if f.name.endswith('.csv')][0]
            
        # 3. Process Data
        df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(final_path)
        metrics = generate_profiling_metrics(df, ds_folder, run_id)
        
        # 4. Write results to existing or newly created table
        results_df = spark.createDataFrame(metrics, schema=spark.table(audit_table).schema)
        results_df.write.format("delta").mode("append").saveAsTable(audit_table)
        
        print(f"Successfully profiled {ds_folder}")
        del df, results_df
        gc.collect()
        
    except Exception as e:
        print(f"Error profiling {ds_folder}: {str(e)}")

### 4. Execution: Batch Dataset Processing
This cell serves as the Control Panel for your profiling tasks. It defines which datasets from your Kaggle ingestion should be analyzed.

Best Practices
- **Modularity**: You can add or remove datasets from the to_process list without changing the underlying code logic.

- **Traceability**: Each run generates a unique profile_run_id, allowing you to group metrics from a single pipeline execution.

In [0]:
import json

def execute_batch_job():
    """
    Retrieves dataset list from job parameters and triggers the profiling loop.
    """
    try:
        # Retrieve JSON array from the task parameter 'datasets_json'
        raw_input = dbutils.widgets.get("datasets_json")
        dataset_list = json.loads(raw_input)
        
        base_vol_path = "/Volumes/data_landing/data_raw"
        
        if dataset_list:
            for ds in dataset_list:
                profile_dataset_dynamically(base_vol_path, ds)
        else:
            print("No datasets found in parameters.")
            
    except Exception as e:
        print(f"Batch execution failed: {str(e)}")

execute_batch_job()

In [0]:
%sql
select * from data_landing.audit.profiling_summary;